# NumCompute quickstart

End-to-end demo of the `numcompute` package.

In [ ]:
%pip install -e ..

In [ ]:
import numpy as np
import numcompute

## Load, inspect, clean and prepare the data

In [ ]:
from numcompute import io as nc_io
from numcompute.preprocessing import (
    StandardScaler, SimpleImputer, OneHotEncoder)

path = "data/penguins.csv"
names = nc_io.read_header(path)
for i, name in enumerate(names):
    print(i, name)

### Load the file
`load_mixed_csv` reads every value as text first, then converts. It returns
two arrays: 
- `X_num` : (numbers, with NaN where a value was missing) 
- `X_cat` : (text categories).

In [ ]:
num_cols = [2, 3, 4, 5]    
cat_cols = [0, 1, 6] 

X_num, X_cat = nc_io.load_mixed_csv(path, num_cols, cat_cols, skip_header=1)

num_names = [names[i] for i in num_cols]
cat_names = [names[i] for i in cat_cols]

print("numeric    :", num_names, X_num.shape, X_num.dtype)
print("categorical:", cat_names, X_cat.shape, X_cat.dtype)

In [ ]:
print(f"{'':>3}" + "".join(f"{n[:15]:>17}" for n in num_names)
      + "".join(f"{n:>12}" for n in cat_names))
for i in range(5):
    print(f"{i:>3}" + "".join(f"{v:>17}" for v in X_num[i])
          + "".join(f"{v:>12}" for v in X_cat[i]))

### Missing data
Missing numerical data must be filled before scaling.

For missing categorical data, such as `sex`, the value is left
blank in the file. The next cell renames it to `unknown` so it reads clearly. It
stays a category of its own, so no penguin is dropped.

In [ ]:
missing = np.isnan(X_num)
print("missing per column:", dict(zip(num_names, missing.sum(axis=0).tolist())))
rows_with_gaps = np.flatnonzero(missing.any(axis=1))
print("rows with a missing number:", rows_with_gaps.size, "of", X_num.shape[0])
print("row numbers:", rows_with_gaps.tolist())

### Rename blank categories
A blank category becomes `unknown`. The counts show how many penguins fall
into each category.

In [ ]:
X_cat = np.where(X_cat == "", "unknown", X_cat)

for j, name in enumerate(cat_names):
    values, counts = np.unique(X_cat[:, j], return_counts=True)
    print(name, dict(zip(values.tolist(), counts.tolist())))


### Fill missing numerical data
`SimpleImputer` replaces each missing number
with the median of its column. The median is a good choice because
extreme values do not affect it much.

In [ ]:
first_gap = int(rows_with_gaps[0])
imputer = SimpleImputer("median").fit(X_num)
filled = imputer.transform(X_num)

print("missing numbers left:", int(np.isnan(filled).sum()))
print("medians used:", dict(zip(num_names, imputer.statistics_.round(1).tolist())))
print(f"row {first_gap} before:", X_num[first_gap])
print(f"row {first_gap} after :", filled[first_gap])

### Scale the numbers
`StandardScaler` rescales each column so it has
mean 0 and standard deviation 1.

In [ ]:
scaler = StandardScaler().fit(filled)
Z = scaler.transform(filled)

print("mean per column:", Z.mean(axis=0).round(3))
print("std per column :", Z.std(axis=0).round(3))
print("row 0 raw   :", filled[0])
print("row 0 scaled:", Z[0].round(2))
print("undo scaling works:", np.allclose(scaler.inverse_transform(Z), filled))

### Encode the categories
`OneHotEncoder` gives every category its own 0/1 column.

In [ ]:
enc = OneHotEncoder().fit(X_cat)
H = enc.transform(X_cat)

for name, cats in zip(cat_names, enc.categories_):
    print(name, "->", cats.tolist())
print("encoded shape:", H.shape)
print("row 0:", X_cat[0].tolist(), "->", H[0])
print("every row has one 1 per column:",
      bool(np.all(H.sum(axis=1) == len(cat_cols))))

### Combine
The scaled numbers and the encoded categories are joined side by side into
one feature matrix.

In [ ]:
features = np.hstack([Z, H])
print("feature matrix:", features.shape)

## Sort, search and rank

This part uses `sort_search` (sorting, top-k, quickselect, binary search) and
`rank` (ranking with ties, percentiles). These functions reject NaN instead of
guessing where it belongs, so this part uses the penguins with all four
measurements.

In [ ]:
from numcompute import sort_search, rank, stats, utils

np.set_printoptions(precision=2, suppress=True)

complete = ~np.isnan(X_num).any(axis=1)
clean = X_num[complete]
species = X_cat[complete, 0]
bill, flipper, mass = clean[:, 0], clean[:, 2], clean[:, 3]

print("penguins with every measurement:", clean.shape[0], "of", X_num.shape[0])

### Top-k
`topk` to find the k largest (or smallest) values in
O(n) without sorting the whole array; only those k are then sorted. It also
returns their positions, so they can be matched to the other columns.

In [ ]:
heaviest, idx = sort_search.topk(mass, 5)
print("5 heaviest (g):", heaviest)
print("species       :", species[idx])

lightest, idx = sort_search.topk(mass, 3, largest=False)
print("3 lightest (g):", lightest)
print("species       :", species[idx])

### Quickselect
`quickselect(values, k)` returns the k-th smallest value (`k=0` is the
minimum) by partitioning around random pivots. That takes O(n) on average,
instead of O(n log n) for a full sort. The answer is checked against sorting.

In [ ]:
n = mass.size
k = n // 2
middle = sort_search.quickselect(mass, k, seed=0)

print(f"k = {k} of {n}: {middle} g")
print("same as sorting:", middle == sort_search.stable_sort(mass)[k])
print("k = 0 (lightest):", sort_search.quickselect(mass, 0, seed=0))
print(f"k = {n - 1} (heaviest):", sort_search.quickselect(mass, n - 1, seed=0))

### Multi-key sort
Rows sorted by flipper length, with body mass breaking ties. The last three
rows all have 178 mm flippers, so mass decides their order. Rows that are equal
on both keys keep their order from the file (the sort is stable).

In [ ]:
by_flipper_then_mass = sort_search.multi_key_sort(clean, [2, 3])

print("".join(f"{n[:15]:>17}" for n in num_names))
for row in by_flipper_then_mass[:6]:
    print("".join(f"{v:>17}" for v in row))

### Binary search
`binary_search` takes a sorted array and returns, for each query, the leftmost
position where it fits and whether it is present. Here that position is also
the number of penguins with a shorter flipper. All queries are searched in one
call.

In [ ]:
flipper_sorted = sort_search.stable_sort(flipper)
queries = np.array([172, 190, 204, 231, 240])
index, exists = sort_search.binary_search(flipper_sorted, queries)

print(f"{'query (mm)':>10}{'position':>10}{'present':>9}")
for q, i, e in zip(queries, np.asarray(index), np.asarray(exists)):
    print(f"{q:>10}{i:>10}{str(e):>9}")

### Ranking with ties
Ranks start at 1 for the lightest penguin. Body mass is recorded in steps of
25 g, so many penguins share a mass. `method` decides what tied values get:
- `average`: the mean of the positions they cover
- `dense`: the same rank, with no ranks skipped after them
- `ordinal`: a different rank each, in the order they appear in the file

In [ ]:
print("different masses:", np.unique(mass).size, "among", mass.size, "penguins")

at_3800 = mass == 3800
print("penguins weighing 3800 g:", int(at_3800.sum()))
for method in ["average", "dense", "ordinal"]:
    print(f"{method:<8}", rank.rank(mass, method)[at_3800])

### Percentiles
`percentile(data, q)` takes `q` from 0 to 100. When a percentile falls between
two data points, `interpolation` picks the answer: `linear` blends the two,
`lower` and `higher` take one of them, and `midpoint` takes their average.

In [ ]:
qs = [25, 50, 75]

print(f"{'':<14}" + "".join(f"{q:>8}" for q in qs))
for method in ["linear", "lower", "higher", "midpoint"]:
    values = np.asarray(rank.percentile(bill, qs, method))
    print(f"{method:<14}" + "".join(f"{v:>8.2f}" for v in values))
print(f"{'np.percentile':<14}" + "".join(f"{v:>8.2f}" for v in np.percentile(bill, qs)))

## Statistics

`stats` functions work on the whole array or along an axis:
- `axis=None`: one number
- `axis=0`: one value per column, shape `(n_features,)`
- `axis=1`: one value per row, shape `(n_samples,)`

`nan_policy` decides what happens with NaN: `propagate` (the result is NaN),
`omit` (NaN is ignored) or `raise` (an error). This part uses the raw `X_num`,
including the rows with missing values.

In [ ]:
from numcompute import stats

print("propagate:", stats.mean(X_num, axis=0))
print("omit     :", stats.mean(X_num, axis=0, nan_policy="omit"))
try:
    stats.mean(X_num, axis=0, nan_policy="raise")
except ValueError as err:
    print("raise    : ValueError:", err)

print()
print("X_num shape:", X_num.shape)
print("axis=None  :", np.shape(stats.mean(X_num, nan_policy="omit")))
print("axis=0     :", stats.mean(X_num, axis=0, nan_policy="omit").shape)
print("axis=1     :", stats.mean(X_num, axis=1, nan_policy="omit").shape)

### Summary of each measurement
`std` uses `ddof=1` here, the sample standard deviation.

In [ ]:
summary = {
    "mean": stats.mean(X_num, axis=0, nan_policy="omit"),
    "median": stats.median(X_num, axis=0, nan_policy="omit"),
    "variance": stats.var(X_num, axis=0, nan_policy="omit"),
    "std (ddof=1)": stats.std(X_num, axis=0, ddof=1, nan_policy="omit"),
    "min": stats.min(X_num, axis=0, nan_policy="omit"),
    "max": stats.max(X_num, axis=0, nan_policy="omit"),
}
print(f"{'':<14}" + "".join(f"{n[:15]:>17}" for n in num_names))
for label, row in summary.items():
    print(f"{label:<14}" + "".join(f"{v:>17.2f}" for v in row))

In [ ]:
for s in np.unique(species):
    m = mass[species == s]
    print(f"{s:<10} n={m.size:>3}   mean mass {stats.mean(m):7.1f} g"
          f"   std {stats.std(m, ddof=1):6.1f} g")

### Histogram
`histogram` returns the count in each bin and the bin edges (one more edge
than bins). NaN is dropped by default, since it belongs in no bin.

In [ ]:
flipper_raw = X_num[:, 2]
counts, edges = stats.histogram(flipper_raw, bins=8)

print("counted:", counts.sum(), "| NaN dropped:", int(np.isnan(flipper_raw).sum()))
for c, lo, hi in zip(counts, edges[:-1], edges[1:]):
    print(f"{lo:6.1f} to {hi:6.1f} mm | {'#' * int(c // 2):<46} {c}")

### Quantiles
`quantile` takes `q` from 0 to 1 (`rank.percentile` uses 0 to 100) and works
along an axis. With three values of `q` and `axis=0` the result is
`(3, n_features)`: one row per `q`.

In [ ]:
Q = stats.quantile(X_num, [0.25, 0.5, 0.75], axis=0, nan_policy="omit")

print("shape:", Q.shape)
print(f"{'':<6}" + "".join(f"{n[:15]:>17}" for n in num_names))
for label, row in zip(["25%", "50%", "75%"], Q):
    print(f"{label:<6}" + "".join(f"{v:>17.2f}" for v in row))
print("bill length matches rank.percentile:",
      np.allclose(Q[:, 0], rank.percentile(bill, [25, 50, 75])))

### Streaming statistics
`RunningStats` updates the mean and variance one chunk at a time (Welford's
method), so the whole dataset never has to be in memory. Here the data is fed
in batches of 50 rows with `utils.batch_iter`, and the result is compared with
the full-array `stats` functions.

In [ ]:
rs = stats.RunningStats()
for batch in utils.batch_iter(clean, 50):
    rs.update(batch)

print("samples seen:", rs.count)
print("mean        :", rs.mean)
print("variance    :", rs.variance())
print("same as stats.mean:", np.allclose(rs.mean, stats.mean(clean, axis=0)))
print("same as stats.var :", np.allclose(rs.variance(), stats.var(clean, axis=0)))